# scLUMEN on zebrafish (ZB)

In [1]:
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import torch

from scLUMEN.datasets import build_wfr_dataframe, load_dataset
from scLUMEN.inference import generate_trajectories
from scLUMEN.refine import refine_decoder
from scLUMEN.runner import (
    ExperimentConfig, evaluate, prepare_latent_data, resolve_device, save_predictions,
    set_seed, train_dynamics, train_latent_model,
)
from scLUMEN.vae import decode_latent

config = ExperimentConfig(
    dataset="zebrafish",
    data_dir="../data/zebrafish",
    output_dir="../results",
    seed=0,
)
device = resolve_device("cpu")
run_dir = config.run_dir
run_dir.mkdir(parents=True, exist_ok=True)

## Load the data

In [2]:
data = load_dataset(config.dataset, config.data_dir)
trajectory = data.trajectory
print("Time points:", data.n_timepoints, "| genes:", data.n_genes)
print("Cells per time point:", [len(cells) for cells in trajectory])
print("Training:", data.train_tps, "| held out:", data.test_tps)
train_frame = build_wfr_dataframe(trajectory, data.train_tps)

Time points: 12 | genes: 2000
Cells per time point: [311, 200, 1158, 1467, 5716, 1026, 4101, 6178, 5442, 7114, 1614, 4404]
Training: (0, 1, 2, 3, 5, 7, 9, 10, 11) | held out: (4, 6, 8)


## Stage 1: latent space

In [3]:
vae = train_latent_model(config, trajectory, train_frame, device)
torch.save(vae.state_dict(), run_dir / "vae.pt")

## Stage 2: unbalanced flow matching

In [4]:
set_seed(config.flow_seed)
latent_train, latent_x0 = prepare_latent_data(config, trajectory, train_frame, vae, device)
dynamics = train_dynamics(config, latent_train, device)
torch.save(dynamics.state_dict(), run_dir / "flow.pt")

## Inference

In [5]:
particles = generate_trajectories(latent_x0, dynamics, device, tuple(range(data.n_timepoints)))
latent, weights = particles["latent"], particles["mass"]
np.savez_compressed(run_dir / "particles.npz", **particles)
print("Latent particles:", latent.shape)

Latent particles: (12, 2000, 10)


## Evaluate the unrefined model

In [6]:
decode_all = lambda model: [np.asarray(decode_latent(model, latent[t], device=device)) for t in range(data.n_timepoints)]
unrefined = pd.DataFrame(evaluate(config, data, trajectory, decode_all(vae), weights, vae, device))
unrefined

,time_point,stage,prop_tv_unweighted,prop_tv_weighted,gene_corr_unweighted,gene_corr_weighted,ot_unweighted,ot_weighted,recon_floor
0,4,E-50,0.143322,0.196021,0.991826,0.984819,406.499448,418.164959,384.444772
1,6,G-60,0.175383,0.103972,0.991826,0.984819,383.226016,392.006523,331.485643
2,8,I-90,0.294046,0.210734,0.991826,0.984819,464.994576,485.739046,387.210236


## Stage 3: mass-aware decoder refinement

In [7]:
refine_decoder(
    vae,
    {t: trajectory[t] for t in data.train_tps},
    {t: latent[t] for t in data.train_tps},
    {t: weights[t] for t in data.train_tps},
    mode="both",
    steps=config.refine_steps,
    batch_size=config.refine_batch_size,
    lr=config.refine_lr,
    blur=config.refine_blur,
    seed=config.seed,
)
torch.save(vae.state_dict(), run_dir / "vae_refined.pt")

In [8]:
predictions = decode_all(vae)
refined = pd.DataFrame(evaluate(config, data, trajectory, predictions, weights, vae, device))
save_predictions(run_dir / "predictions.npz", data, predictions, weights)
metrics = pd.concat([unrefined.assign(decoder="none"), refined.assign(decoder="both")])
metrics.insert(0, "seed", config.seed)
metrics.insert(0, "dataset", config.dataset)
metrics.to_csv(run_dir / "metrics.csv", index=False)
refined

,time_point,stage,prop_tv_unweighted,prop_tv_weighted,gene_corr_unweighted,gene_corr_weighted,ot_unweighted,ot_weighted,recon_floor
0,4,E-50,0.116822,0.192240,0.993042,0.992248,407.220131,416.771408,387.429087
1,6,G-60,0.180627,0.132494,0.993042,0.992248,370.941445,371.913915,336.823030
2,8,I-90,0.230115,0.084145,0.993042,0.992248,443.811382,437.327995,399.491138


In [9]:
columns = [c for c in metrics if c.startswith(("ot_", "prop_", "gene_corr"))]
metrics.groupby("decoder")[columns].mean().round(3)

,prop_tv_unweighted,prop_tv_weighted,gene_corr_unweighted,gene_corr_weighted,ot_unweighted,ot_weighted
decoder,,,,,,
both,0.176,0.136,0.993,0.992,407.324,408.671
none,0.204,0.170,0.992,0.985,418.240,431.970
